In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import scanpy.external as sce
import tqdm as tqdm
import scallop as sl
import sys
sys.path.append('C:/Users/yang2/decibel')
import module.decibel as dcb
from tqdm.notebook import tqdm
from scipy import sparse
from scipy.spatial.distance import euclidean

# 1. TMS bone marrow cells

In [ ]:
adata_TMS_marrow = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Results/TMS/TMS_marrow/TMS_marrow.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_TMS_marrow, target_sum=1e4)
sc.pp.log1p(adata_TMS_marrow)

In [ ]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_TMS_marrow.obs.rename(columns={'celltype': 'cell_type'}, inplace=True)
adata_TMS_marrow = dcb.distance_to_celltype_mean(adata_TMS_marrow, 'age')

In [ ]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_TMS_marrow.X):
    X = adata_TMS_marrow.X.toarray()
else:
    X = adata_TMS_marrow.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_TMS_marrow.X = sparse.csr_matrix(X)
adata_TMS_marrow.var['means'] = pd.Series(np.asarray(adata_TMS_marrow.X.mean(axis=0)).ravel(), index=adata_TMS_marrow.var_names)

# Sort genes according to their mean expression
gene_order = adata_TMS_marrow.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_TMS_marrow)
least_variable = dcb.get_least_variable(adata_TMS_marrow, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_TMS_marrow[:, least_variable].X.mean(axis=0)).ravel()

adata_TMS_marrow.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_TMS_marrow[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_TMS_marrow.shape[0]))]

In [ ]:
# Method 3: Scallop
scal_TMS_marrow = sl.Scallop(adata_TMS_marrow)
score = sl.tl.getScore(scal_TMS_marrow, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_TMS_marrow.obs['scallop_score'] = score

In [ ]:
# Save outputs
adata_TMS_marrow.obs.to_csv("adata_TMS_marrow_metadata.csv")

# 2. Liver hepatocytes

In [ ]:
adata_liver = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Data/C57BL6J_liver_hippocampus/liver_data_hep_only.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_liver, target_sum=1e4)
sc.pp.log1p(adata_liver)

In [ ]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_liver.obs.rename(columns={'annotation': 'cell_type'}, inplace=True)
adata_liver = dcb.distance_to_celltype_mean(adata_liver, 'age')

In [ ]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_liver.X):
    X = adata_liver.X.toarray()
else:
    X = adata_liver.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_liver.X = sparse.csr_matrix(X)
adata_liver.var['means'] = pd.Series(np.asarray(adata_liver.X.mean(axis=0)).ravel(), index=adata_liver.var_names)

# Sort genes according to their mean expression
gene_order = adata_liver.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_liver)
least_variable = dcb.get_least_variable(adata_liver, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_liver[:, least_variable].X.mean(axis=0)).ravel()

adata_liver.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_liver[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_liver.shape[0]))]

In [ ]:
# Method 3: Scallop
scal_liver = sl.Scallop(adata_liver)
score = sl.tl.getScore(scal_liver, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_liver.obs['scallop_score'] = score

In [ ]:
# Save outputs
adata_liver.obs.to_csv("adata_liver_metadata.csv")

# 3. Kidney PT cells

In [ ]:
adata_kidney_PT_only = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/RAGE24/cellranger_flowcell_combine/kidney_aging_samples_integrated_PT_only.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_kidney_PT_only, target_sum=1e4)
sc.pp.log1p(adata_kidney_PT_only)

In [ ]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_kidney_PT_only.obs.rename(columns={'celltype_refined2': 'cell_type'}, inplace=True)
adata_kidney_PT_only = dcb.distance_to_celltype_mean(adata_kidney_PT_only, 'age_wks')

In [ ]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_kidney_PT_only.X):
    X = adata_kidney_PT_only.X.toarray()
else:
    X = adata_kidney_PT_only.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_kidney_PT_only.X = sparse.csr_matrix(X)
adata_kidney_PT_only.var['means'] = pd.Series(np.asarray(adata_kidney_PT_only.X.mean(axis=0)).ravel(), index=adata_kidney_PT_only.var_names)

# Sort genes according to their mean expression
gene_order = adata_kidney_PT_only.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_kidney_PT_only)
least_variable = dcb.get_least_variable(adata_kidney_PT_only, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_kidney_PT_only[:, least_variable].X.mean(axis=0)).ravel()

adata_kidney_PT_only.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_kidney_PT_only[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_kidney_PT_only.shape[0]))]

In [ ]:
# Method 3: Scallop
scal_kidney_PT_only = sl.Scallop(adata_kidney_PT_only)
score = sl.tl.getScore(scal_kidney_PT_only, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_kidney_PT_only.obs['scallop_score'] = score

In [ ]:
# Save outputs
adata_kidney_PT_only.obs.to_csv("adata_kidney_PT_only_metadata.csv")

# 4. Human kidney tubular cells

In [ ]:
adata_human_kidney= sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/Human_kidney/human_kidney.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_human_kidney, target_sum=1e4)
sc.pp.log1p(adata_human_kidney)

In [ ]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_human_kidney.obs.rename(columns={'celltype': 'cell_type'}, inplace=True)
adata_human_kidney = dcb.distance_to_celltype_mean(adata_human_kidney, 'age_bin')

In [ ]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_human_kidney.X):
    adata_human_kidney.X = adata_human_kidney.X.tocsr()
    adata_human_kidney.X.data[np.isnan(adata_human_kidney.X.data)] = 0
    adata_human_kidney.X.data[np.isinf(adata_human_kidney.X.data)] = 0
else:
    adata_human_kidney.X[np.isnan(adata_human_kidney.X)] = 0
    adata_human_kidney.X[np.isinf(adata_human_kidney.X)] = 0

adata_human_kidney.var['means'] = pd.Series(np.asarray(adata_human_kidney.X.mean(axis=0)).ravel(), index=adata_human_kidney.var_names)

# Sort genes according to their mean expression
gene_order = adata_human_kidney.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_human_kidney)
least_variable = dcb.get_least_variable(adata_human_kidney, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_human_kidney[:, least_variable].X.mean(axis=0)).ravel()

adata_human_kidney.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_human_kidney[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_human_kidney.shape[0]))]

In [ ]:
# Method 3: Scallop
scal_human_kidney = sl.Scallop(adata_human_kidney)
score = sl.tl.getScore(scal_human_kidney, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_human_kidney.obs['scallop_score'] = score

In [ ]:
# Save outputs
adata_human_kidney.obs.to_csv("adata_human_kidney_metadata.csv")

# 5. Human bone marrow cells

In [ ]:
adata_human_BM = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/Human_BM/human_BM_Oetjen.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_human_BM, target_sum=1e4)
sc.pp.log1p(adata_human_BM)

In [ ]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_human_BM.obs.rename(columns={'celltype': 'cell_type'}, inplace=True)
adata_human_BM = dcb.distance_to_celltype_mean(adata_human_BM, 'age_group')

In [ ]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_human_BM.X):
    X = adata_human_BM.X.toarray()
else:
    X = adata_human_BM.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_human_BM.X = sparse.csr_matrix(X)
adata_human_BM.var['means'] = pd.Series(np.asarray(adata_human_BM.X.mean(axis=0)).ravel(), index=adata_human_BM.var_names)

# Sort genes according to their mean expression
gene_order = adata_human_BM.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_human_BM)
least_variable = dcb.get_least_variable(adata_human_BM, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_human_BM[:, least_variable].X.mean(axis=0)).ravel()

adata_human_BM.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_human_BM[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_human_BM.shape[0]))]

In [ ]:
# Method 3: Scallop
scal_human_BM = sl.Scallop(adata_human_BM)
score = sl.tl.getScore(scal_human_BM, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_human_BM.obs['scallop_score'] = score

In [ ]:
# Save outputs
adata_human_BM.obs.to_csv("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/Human_BM/adata_human_BM_Oetjen_metadata.csv")

# 6. Mouse MSCs

In [ ]:
adata_mouse_SC = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/Mouse_MSC/mouse_MSC_SC.h5ad")
adata_mouse_FAP = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/Mouse_MSC/mouse_MSC_FAP.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_mouse_SC, target_sum=1e4)
sc.pp.log1p(adata_mouse_SC)
sc.pp.normalize_total(adata_mouse_FAP, target_sum=1e4)
sc.pp.log1p(adata_mouse_FAP)

In [ ]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_mouse_SC = dcb.distance_to_celltype_mean(adata_mouse_SC, 'condition')
adata_mouse_FAP = dcb.distance_to_celltype_mean(adata_mouse_FAP, 'condition')

In [ ]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_mouse_SC.X):
    X = adata_mouse_SC.X.toarray()
else:
    X = adata_mouse_SC.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_mouse_SC.X = sparse.csr_matrix(X)
adata_mouse_SC.var['means'] = pd.Series(np.asarray(adata_mouse_SC.X.mean(axis=0)).ravel(), index=adata_mouse_SC.var_names)

# Sort genes according to their mean expression
gene_order = adata_mouse_SC.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_mouse_SC)
least_variable = dcb.get_least_variable(adata_mouse_SC, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_mouse_SC[:, least_variable].X.mean(axis=0)).ravel()

adata_mouse_SC.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_mouse_SC[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_mouse_SC.shape[0]))]

if sparse.issparse(adata_mouse_FAP.X):
    X = adata_mouse_FAP.X.toarray()
else:
    X = adata_mouse_FAP.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_mouse_FAP.X = sparse.csr_matrix(X)
adata_mouse_FAP.var['means'] = pd.Series(np.asarray(adata_mouse_FAP.X.mean(axis=0)).ravel(), index=adata_mouse_FAP.var_names)

# Sort genes according to their mean expression
gene_order = adata_mouse_FAP.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_mouse_FAP)
least_variable = dcb.get_least_variable(adata_mouse_FAP, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_mouse_FAP[:, least_variable].X.mean(axis=0)).ravel()

adata_mouse_FAP.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_mouse_FAP[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_mouse_FAP.shape[0]))]

In [ ]:
# Method 3: Scallop
scal_mouse_SC = sl.Scallop(adata_mouse_SC)
score = sl.tl.getScore(scal_mouse_SC, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_mouse_SC.obs['scallop_score'] = score

scal_mouse_FAP = sl.Scallop(adata_mouse_FAP)
score = sl.tl.getScore(scal_mouse_FAP, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_mouse_FAP.obs['scallop_score'] = score

In [ ]:
# Save outputs
adata_mouse_SC.obs.to_csv("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/Mouse_MSC/adata_Mouse_MSC_SC_metadata.csv")
adata_mouse_FAP.obs.to_csv("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/Mouse_MSC/adata_Mouse_MSC_FAP_metadata.csv")

# 7. Mouse aortic cells

In [ ]:
adata_mouse_VSMC = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/Mouse_VSMC/mouse_VSMC_aorta.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_mouse_VSMC, target_sum=1e4)
sc.pp.log1p(adata_mouse_VSMC)

In [ ]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_mouse_VSMC.obs.rename(columns={'celltype': 'cell_type'}, inplace=True)
adata_mouse_VSMC = dcb.distance_to_celltype_mean(adata_mouse_VSMC, 'condition')

In [ ]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_mouse_VSMC.X):
    X = adata_mouse_VSMC.X.toarray()
else:
    X = adata_mouse_VSMC.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_mouse_VSMC.X = sparse.csr_matrix(X)
adata_mouse_VSMC.var['means'] = pd.Series(np.asarray(adata_mouse_VSMC.X.mean(axis=0)).ravel(), index=adata_mouse_VSMC.var_names)

# Sort genes according to their mean expression
gene_order = adata_mouse_VSMC.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_mouse_VSMC)
least_variable = dcb.get_least_variable(adata_mouse_VSMC, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_mouse_VSMC[:, least_variable].X.mean(axis=0)).ravel()

adata_mouse_VSMC.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_mouse_VSMC[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_mouse_VSMC.shape[0]))]


In [ ]:
# Method 3: Scallop
scal_mouse_VSMC = sl.Scallop(adata_mouse_VSMC)
score = sl.tl.getScore(scal_mouse_VSMC, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_mouse_VSMC.obs['scallop_score'] = score

In [ ]:
# Save outputs
adata_mouse_VSMC.obs.to_csv("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/Mouse_VSMC/adata_Mouse_VSMC_metadata.csv")

# 8. Cancerous human T cells

In [ ]:
adata_T_cell = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/T_cell_exhaustion/cancerous_t_cell.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_T_cell, target_sum=1e4)
sc.pp.log1p(adata_T_cell)

In [ ]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_T_cell = dcb.distance_to_celltype_mean(adata_T_cell, 'Condition')

In [ ]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_T_cell.X):
    X = adata_T_cell.X.toarray()
else:
    X = adata_T_cell.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_T_cell.X = sparse.csr_matrix(X)
adata_T_cell.var['means'] = pd.Series(np.asarray(adata_T_cell.X.mean(axis=0)).ravel(), index=adata_T_cell.var_names)

# Sort genes according to their mean expression
gene_order = adata_T_cell.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_T_cell)
least_variable = dcb.get_least_variable(adata_T_cell, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_T_cell[:, least_variable].X.mean(axis=0)).ravel()

adata_T_cell.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_T_cell[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_T_cell.shape[0]))]


In [ ]:
# Method 3: Scallop
scal_T_cell = sl.Scallop(adata_T_cell)
score = sl.tl.getScore(scal_T_cell, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_T_cell.obs['scallop_score'] = score

In [ ]:
# Save outputs
adata_T_cell.obs.to_csv("S:/Penn Dropbox/Eddie Yang/Aging/Scripts/T_cell_exhaustion/adata_T_cell_metadata.csv")